# Kroma voice engine on a free Colab GPU

Runs Kroma's voice engine on Google's free T4 GPU, several times faster than a laptop.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. The last cell prints an **address** and a **token**: paste them in Kroma → Settings → Voice Engine → Google Colab.

Keep this tab open while you use Kroma. The first run downloads the voice model (about 5 GB).

In [ ]:
# 1) Get Kroma's engine (public, no account needed)
import os, subprocess
if not os.path.exists('/content/kroma'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Heang-Dev/KROMA.STUDIO.git', '/content/kroma'], check=True)
else:
    subprocess.run(['git', '-C', '/content/kroma', 'pull', '-q'])
%cd /content/kroma/engine

In [ ]:
# 2) Install (about 3-5 minutes)
!apt-get -qq install -y ffmpeg fonts-khmeros > /dev/null
!pip -q install -r requirements.txt

In [ ]:
# 3) Start the engine and a public HTTPS address (Cloudflare quick tunnel, no account needed)
import os, re, secrets, subprocess
os.chdir('/content/kroma/engine')
os.environ.setdefault('KVS_ENGINE_TOKEN', secrets.token_urlsafe(24))
os.environ.setdefault('KVS_OUTPUTS_DIR', '/content/kroma-outputs')
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
engine = subprocess.Popen(['python', 'server.py', '--host', '127.0.0.1', '--port', '7861'], stdout=open('/content/engine.log', 'w'), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:7861'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('Paste these in Kroma → Settings → Voice Engine → Google Colab:\n')
print(f'Address:  {url}')
print(f"Token:    {os.environ['KVS_ENGINE_TOKEN']}")
print('\nKeep this tab open while you use Kroma. Engine log: /content/engine.log')